# 🌍 Income Inequality Across the Globe
**Data Analysis & Exploration**  
Source: UNDP Human Development Reports (2010–2021)

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.float_format", "{:.2f}".format)
pd.set_option("display.max_columns", 20)

## 2. Load Raw Data

In [ ]:
df = pd.read_csv("Inequality in Income.csv")
print(f"Shape: {df.shape}")
print(f"Countries: {df['Country'].nunique()}")
df.head()

## 3. Data Cleaning

### 3.1 Inspect Missing Values

In [ ]:
print("Null counts per column:")
print(df.isnull().sum())
print(f"\nTotal cells: {df.size}")
print(f"Missing cells: {df.isnull().sum().sum()} ({df.isnull().sum().sum()/df.size*100:.1f}%)")

### 3.2 Remove Duplicates

In [ ]:
before = len(df)
df = df.drop_duplicates()
print(f"Rows before: {before} | After: {len(df)} | Removed: {before - len(df)}")

### 3.3 Export Cleaned Wide-Format to Excel

The cleaned data is saved as `.xlsx`. Power Query was then used in Excel to **unpivot (melt) the year columns vertically** into a long format for better analysis.

In [ ]:
df.to_excel("income_inequality.xlsx", index=False)
print("Exported to income_inequality.xlsx")

## 4. Transform to Long Format
*(Python equivalent of the Power Query unpivot step)*

Instead of wide format (one column per year), we reshape so each row represents **one country × one year**. This makes time-series analysis and grouping far easier.

In [ ]:
year_cols = [c for c in df.columns if "Inequality in income" in c]

df_long = df.melt(
    id_vars=["ISO3", "Country", "Continent", "Hemisphere",
             "Human Development Groups", "UNDP Developing Regions", "HDI Rank (2021)"],
    value_vars=year_cols,
    var_name="Year",
    value_name="Inequality_in_Income"
)

# Extract numeric year
df_long["Year"] = df_long["Year"].str.extract(r"(\d{4})").astype(int)

# Track excluded countries before dropping
countries_all = set(df_long["Country"])
df_long = df_long.dropna(subset=["Inequality_in_Income"]).reset_index(drop=True)
countries_kept = set(df_long["Country"])
excluded = sorted(countries_all - countries_kept)

print(f"Long format shape: {df_long.shape}")
print(f"Countries with data: {df_long['Country'].nunique()}")
print(f"Countries excluded (no income data): {len(excluded)}")
print(f"Excluded: {excluded}")
df_long.head(10)

In [ ]:
# Save long-format CSV for reproducibility
df_long.to_csv("income_inequality_long.csv", index=False)
print("Saved income_inequality_long.csv")

## 5. Exploratory Data Analysis

### 5.1 Global Average Inequality Over Time

In [ ]:
yearly_avg = df_long.groupby("Year")["Inequality_in_Income"].mean().reset_index()
yearly_avg.columns = ["Year", "Global Avg Inequality"]
yearly_avg["Global Avg Inequality"] = yearly_avg["Global Avg Inequality"].round(2)
first = yearly_avg["Global Avg Inequality"].iloc[0]
last = yearly_avg["Global Avg Inequality"].iloc[-1]
print(yearly_avg.to_string(index=False))
print(f"\nOverall change: {first:.2f} -> {last:.2f} ({last - first:+.2f} pp)")

### 5.2 Average Inequality by Continent

In [ ]:
continent_stats = df_long.groupby("Continent")["Inequality_in_Income"].agg(["mean","median","std","count"])
continent_stats.columns = ["Mean","Median","Std Dev","Data Points"]
continent_stats = continent_stats.sort_values("Mean", ascending=False).round(2)
print(continent_stats)

### 5.3 Average Inequality by Human Development Group

In [ ]:
hdi_order = ["Low", "Medium", "High", "Very High"]
hdi_stats = df_long.groupby("Human Development Groups")["Inequality_in_Income"].agg(["mean","median","std"])
hdi_stats.columns = ["Mean","Median","Std Dev"]
hdi_stats = hdi_stats.reindex([h for h in hdi_order if h in hdi_stats.index]).round(2)
print(hdi_stats)

### 5.4 Most & Least Unequal Countries (2021)

In [ ]:
latest = df_long[df_long["Year"] == 2021].copy()

print("Top 10 Most Unequal Countries (2021):")
most = latest.nlargest(10, "Inequality_in_Income")[["Country","Continent","Human Development Groups","Inequality_in_Income"]]
most["Inequality_in_Income"] = most["Inequality_in_Income"].round(2)
print(most.to_string(index=False))

print("\nTop 10 Least Unequal Countries (2021):")
least = latest.nsmallest(10, "Inequality_in_Income")[["Country","Continent","Human Development Groups","Inequality_in_Income"]]
least["Inequality_in_Income"] = least["Inequality_in_Income"].round(2)
print(least.to_string(index=False))

### 5.5 Countries with Largest Change (2010 to 2021)

In [ ]:
pivot = df_long[df_long["Year"].isin([2010, 2021])].pivot_table(
    index=["Country","Continent"], columns="Year", values="Inequality_in_Income"
).dropna()
pivot.columns = ["Yr_2010","Yr_2021"]
pivot["Change"] = (pivot["Yr_2021"] - pivot["Yr_2010"]).round(2)

print("Most Improved (largest reduction in inequality):")
print(pivot.nsmallest(10, "Change")[["Yr_2010","Yr_2021","Change"]].to_string())

print("\nMost Worsened (largest increase in inequality):")
print(pivot.nlargest(10, "Change")[["Yr_2010","Yr_2021","Change"]].to_string())

### 5.6 Overall Summary Statistics

In [ ]:
print("Inequality_in_Income — Summary:")
print(df_long["Inequality_in_Income"].describe().round(2))

## 6. Key Findings

| Finding | Detail |
|---|---|
| **Global trend** | Average income inequality declined from **24.32** (2010) to **22.81** (2021) |
| **Most unequal region** | Americas (avg 31.46), followed by Africa (avg 28.98) |
| **Least unequal region** | Europe (avg 15.22) |
| **HDI correlation** | Very High HDI countries have significantly lower inequality (avg 17.95 vs 27.43 for Medium HDI) |
| **Most unequal country (2021)** | South Africa (56.99) |
| **Least unequal country (2021)** | Slovenia (8.31) |
| **Data gaps** | 24 countries had no income inequality data at all |
| **Largest improvement** | Several countries in ECA region showed consistent reduction |

> **Note:** The Inequality in Income metric used here is the **Atkinson index**, measuring the percentage loss in human development due to inequality in income distribution (UNDP methodology).

## 7. Output Files

| File | Description |
|---|---|
| `income_inequality.xlsx` | Cleaned wide-format data |
| `income_inequality_long.csv` | Long-format (unpivoted by year), ready for analysis |
| `Income_Inequality.twb` | Tableau visualization workbook |